# 🏠 Irish Property Deal Scorer
Trains a model on PPR data and scores any property against comparable transactions.

**Output:** A deal score telling the user if a property is underpriced, fair, or overpriced relative to the market.

In [1]:
# Install dependencies if needed
# !pip install pandas scikit-learn xgboost joblib

In [2]:
import pandas as pd
import numpy as np
import re
import joblib
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

print('Libraries loaded ✅')

Libraries loaded ✅


## 1. Load & Clean PPR Data

In [3]:
# ── CHANGE THIS PATH to your local PPR CSV ──
PPR_PATH = 'PPR-ALL.csv'

df = pd.read_csv(PPR_PATH, encoding='latin-1')
print('Raw shape:', df.shape)
print(df.columns.tolist())
df.head(3)

Raw shape: (772832, 9)
['Date of Sale (dd/mm/yyyy)', 'Address', 'County', 'Eircode', 'Price (\x80)', 'Not Full Market Price', 'VAT Exclusive', 'Description of Property', 'Property Size Description']


C:\Users\WAFI\AppData\Local\Temp\ipykernel_4436\3328516195.py:4: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(PPR_PATH, encoding='latin-1')


,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description
0,01/01/2010,"5 Braemor Drive, Churchtown, Co.Dublin",Dublin,NaN,"343,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
1,03/01/2010,"134 Ashewood Walk, Summerhill Lane, Portlaoise",Laois,NaN,"185,000.00",No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...
2,04/01/2010,"1 Meadow Avenue, Dundrum, Dublin 14",Dublin,NaN,"438,500.00",No,No,Second-Hand Dwelling house /Apartment,NaN


In [4]:
# Rename columns to standard names (adjust if yours differ)
df.columns = [
    'date_of_sale', 'address', 'postal_code', 'county',
    'price', 'not_full_market_price', 'vat_exclusive',
    'description', 'property_size_desc'
]

# Clean price
df['price'] = df['price'].astype(str).str.replace('[€,]', '', regex=True).str.strip()
df['price'] = pd.to_numeric(df['price'], errors='coerce')

# Parse year
df['date_of_sale'] = pd.to_datetime(df['date_of_sale'], dayfirst=True, errors='coerce')
df['year'] = df['date_of_sale'].dt.year

# Extract micro-area from address (first part before comma)
df['micro_area'] = df['address'].astype(str).apply(
    lambda x: x.split(',')[0].strip().title()
)

# Clean county
df['county'] = df['county'].astype(str).str.strip().str.title()

# Filter: realistic price range, recent years, full market price only
df = df[
    (df['price'] >= 30_000) &
    (df['price'] <= 5_000_000) &
    (df['year'] >= 2010) &
    (df['not_full_market_price'].astype(str).str.strip() == 'No')
].copy()

print('Cleaned shape:', df.shape)
df[['county', 'micro_area', 'price', 'year', 'description']].head(5)

Cleaned shape: (0, 11)


,county,micro_area,price,year,description


## 2. Feature Engineering

In [5]:
# Encode categorical features
le_county = LabelEncoder()
le_desc = LabelEncoder()
le_micro = LabelEncoder()

df['county_enc'] = le_county.fit_transform(df['county'].fillna('Unknown'))
df['desc_enc'] = le_desc.fit_transform(df['description'].fillna('Unknown'))
df['micro_enc'] = le_micro.fit_transform(df['micro_area'].fillna('Unknown'))

# Log price as target (improves model performance on skewed price distributions)
df['log_price'] = np.log1p(df['price'])

FEATURES = ['county_enc', 'micro_enc', 'desc_enc', 'year']
TARGET = 'log_price'

model_df = df[FEATURES + [TARGET]].dropna()
print('Model dataset shape:', model_df.shape)

Model dataset shape: (0, 5)


## 3. Train the Model

In [6]:
X = model_df[FEATURES]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred))
r2 = r2_score(y_test, y_pred)

print(f'✅ Model trained')
print(f'   MAE:  €{mae:,.0f}')
print(f'   R²:   {r2:.3f}')

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

## 4. Save Model & Encoders

In [ ]:
joblib.dump(model, 'deal_scorer_model.pkl')
joblib.dump(le_county, 'le_county.pkl')
joblib.dump(le_desc, 'le_desc.pkl')
joblib.dump(le_micro, 'le_micro.pkl')

# Save known values for dropdowns
import json
known = {
    'counties': sorted(le_county.classes_.tolist()),
    'descriptions': sorted(le_desc.classes_.tolist()),
}
with open('known_values.json', 'w') as f:
    json.dump(known, f)

print('✅ Model and encoders saved')
print('Files: deal_scorer_model.pkl, le_county.pkl, le_desc.pkl, le_micro.pkl, known_values.json')

## 5. Deal Scorer Function

In [ ]:
def score_deal(county: str, area: str, asking_price: float, 
               description: str = 'Second-Hand Dwelling house /Duplex', 
               year: int = 2024):
    """
    Score a property deal against the model's prediction.
    
    Returns a dict with:
      - predicted_price: what the model thinks it's worth
      - asking_price: what they're asking
      - difference_pct: % above/below market
      - verdict: STRONG BUY / GOOD DEAL / FAIR / OVERPRICED / AVOID
      - comparable_sales: recent sales in same area for context
    """
    # Handle unseen counties/areas gracefully
    county_title = county.strip().title()
    area_title = area.strip().title()
    desc_clean = description.strip()

    if county_title not in le_county.classes_:
        return {'error': f'County "{county}" not found in training data. Try: {le_county.classes_[:5].tolist()}'}
    
    # For unknown micro-areas, fall back to county median
    if area_title not in le_micro.classes_:
        # Use most common area in that county as fallback
        county_areas = df[df['county'] == county_title]['micro_area'].value_counts()
        area_title = county_areas.index[0] if len(county_areas) > 0 else le_micro.classes_[0]
        fallback_note = f'Area not found — using county average as benchmark'
    else:
        fallback_note = None

    if desc_clean not in le_desc.classes_:
        desc_clean = le_desc.classes_[0]

    county_enc = le_county.transform([county_title])[0]
    micro_enc = le_micro.transform([area_title])[0]
    desc_enc = le_desc.transform([desc_clean])[0]

    X_input = pd.DataFrame([{
        'county_enc': county_enc,
        'micro_enc': micro_enc,
        'desc_enc': desc_enc,
        'year': year
    }])

    log_pred = model.predict(X_input)[0]
    predicted_price = np.expm1(log_pred)

    diff_pct = ((asking_price - predicted_price) / predicted_price) * 100

    if diff_pct <= -15:
        verdict = '🟢 STRONG BUY'
        explanation = f'This property is {abs(diff_pct):.1f}% below market value — significantly underpriced.'
    elif diff_pct <= -5:
        verdict = '🟢 GOOD DEAL'
        explanation = f'Priced {abs(diff_pct):.1f}% below market — good value for this area.'
    elif diff_pct <= 5:
        verdict = '🟡 FAIR'
        explanation = f'Priced within 5% of market rate — fair value.'
    elif diff_pct <= 15:
        verdict = '🟠 OVERPRICED'
        explanation = f'Asking price is {diff_pct:.1f}% above market — negotiate down.'
    else:
        verdict = '🔴 AVOID'
        explanation = f'Asking price is {diff_pct:.1f}% above market — significantly overpriced.'

    # Comparable sales
    comparables = df[
        (df['county'] == county_title) &
        (df['year'] >= 2022)
    ]['price'].describe()

    result = {
        'verdict': verdict,
        'explanation': explanation,
        'asking_price': f'€{asking_price:,.0f}',
        'predicted_market_value': f'€{predicted_price:,.0f}',
        'difference': f'{diff_pct:+.1f}%',
        'county_median_2022_2024': f'€{comparables["50%"]:,.0f}',
        'county_sales_count_2022_2024': int(comparables['count']),
    }
    if fallback_note:
        result['note'] = fallback_note
    return result

print('✅ score_deal() function ready')

## 6. Test It

In [ ]:
# Test examples
test_cases = [
    ('Dublin', 'Ballymun', 320_000),
    ('Dublin', 'Blackrock', 750_000),
    ('Cork', 'Douglas', 420_000),
    ('Galway', 'Salthill', 380_000),
]

for county, area, price in test_cases:
    result = score_deal(county, area, price)
    print(f'\n📍 {area}, {county} — Asking €{price:,}')
    for k, v in result.items():
        print(f'   {k}: {v}')

## 7. Flask Route (copy into your app.py)

In [ ]:
flask_code = '''
import joblib
import json
import numpy as np
import pandas as pd

# Load at startup
deal_model = joblib.load('deal_scorer_model.pkl')
deal_le_county = joblib.load('le_county.pkl')
deal_le_desc = joblib.load('le_desc.pkl')
deal_le_micro = joblib.load('le_micro.pkl')
with open('known_values.json') as f:
    known_values = json.load(f)

@app.route('/deal-checker', methods=['GET', 'POST'])
def deal_checker():
    result = None
    if request.method == 'POST':
        county = request.form.get('county', '').strip().title()
        area = request.form.get('area', '').strip().title()
        asking_price = float(request.form.get('asking_price', 0))
        description = request.form.get('description', 'Second-Hand Dwelling house /Duplex')

        # Encode
        try:
            county_enc = deal_le_county.transform([county])[0]
        except:
            county_enc = 0
        try:
            micro_enc = deal_le_micro.transform([area])[0]
        except:
            micro_enc = 0
        try:
            desc_enc = deal_le_desc.transform([description])[0]
        except:
            desc_enc = 0

        X = pd.DataFrame([{'county_enc': county_enc, 'micro_enc': micro_enc,
                            'desc_enc': desc_enc, 'year': 2024}])
        log_pred = deal_model.predict(X)[0]
        predicted = np.expm1(log_pred)
        diff_pct = ((asking_price - predicted) / predicted) * 100

        if diff_pct <= -15: verdict = 'STRONG BUY'
        elif diff_pct <= -5: verdict = 'GOOD DEAL'
        elif diff_pct <= 5: verdict = 'FAIR'
        elif diff_pct <= 15: verdict = 'OVERPRICED'
        else: verdict = 'AVOID'

        result = {
            'verdict': verdict,
            'predicted': f"€{predicted:,.0f}",
            'asking': f"€{asking_price:,.0f}",
            'diff': f"{diff_pct:+.1f}%"
        }

    return render_template('deal_checker.html',
                           result=result,
                           counties=known_values['counties'],
                           descriptions=known_values['descriptions'])
'''

print(flask_code)

In [ ]:
PPR_PATH = 'PPR-ALL.csv'